Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier

# Binarize the labels for a multi-class problem
y_bin = label_binarize(y_test_fixed, classes=np.unique(y_train_fixed))
n_classes = y_bin.shape[1]

# Learn to predict each class against the other using a OneVsRest strategy
classifier = OneVsRestClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
y_score = classifier.fit(X_train_fixed, y_train_fixed).predict_proba(X_test_fixed)

# Initialize dictionaries to hold true positive rates and false positive rates
fpr = dict()
tpr = dict()
roc_auc = dict()

# Compute ROC curve and ROC area for each class
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Compute micro-average ROC curve and ROC area
fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), y_score.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Aggregate all false positive rates
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

# Interpolate all ROC curves at these points
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

# Average it and compute AUC
mean_tpr /= n_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# Plotting both curves
plt.figure(figsize=(8, 6))
plt.plot(fpr["micro"], tpr["micro"],
         label='Micro-average ROC curve (area = {0:0.2f})'.format(roc_auc["micro"]),
         color='deeppink', linestyle=':', linewidth=4)

plt.plot(fpr["macro"], tpr["macro"],
         label='Macro-average ROC curve (area = {0:0.2f})'.format(roc_auc["macro"]),
         color='navy', linestyle=':', linewidth=4)

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Micro and Macro Average ROC Curves')
plt.legend(loc="lower right")
plt.show()


Brier Score Calculation

In [ ]:
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import brier_score_loss

# Step 1: Convert y_test_fixed to one-hot encoding if it's not already
lb = LabelBinarizer()
lb.fit(y_test_fixed)
y_test_one_hot = lb.transform(y_test_fixed)

# Step 2: Obtain probabilistic predictions for all classes
y_proba_fixed = clf_fixed.predict_proba(X_test_fixed)

# Step 3: Calculate the Brier score for each class and average
brier_scores = [brier_score_loss(y_test_one_hot[:, i], y_proba_fixed[:, i])
                for i in range(y_proba_fixed.shape[1])]
average_brier_score = np.mean(brier_scores)

print(f"Average Brier score for the multiclass classification: {average_brier_score}")


ECE Calculation

In [ ]:
def calculate_ece(y_true, y_pred_probs, n_bins=10):
    y_true = torch.tensor(y_true)
    y_pred_probs = torch.tensor(y_pred_probs)
    bin_boundaries = torch.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]

    ece = torch.tensor(0.0)
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (y_pred_probs > bin_lower) & (y_pred_probs <= bin_upper)
        prop_in_bin = in_bin.float().mean()
        if prop_in_bin.item() > 0:
            # Corrected accuracy calculation: Ensure indexing and comparison are correctly applied
            accurate_preds = (y_pred_probs[in_bin] >= 0.5).float() == y_true[in_bin]
            accuracy_in_bin = accurate_preds.float().mean()

            avg_confidence_in_bin = y_pred_probs[in_bin].mean()
            ece += torch.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    return ece.item()

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_test_encoded = label_encoder.fit_transform(y_test_fixed)


In [ ]:
ece = calculate_ece(y_test_encoded, y_pred_probs[:, 1], n_bins=10)
print(f"ECE: {ece}")


ROC-AUC Curve

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import CountVectorizer

def preprocess_data_fixed_length(directory):
    all_features = []
    labels = []

    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        with open(file_path, 'r') as json_file:
            data = json.load(json_file)
            feature_str = ' '.join([f'node{node_id}_feature{feature_value}' for node_id, feature_value in data['features'].items()])
            all_features.append(feature_str)
            labels.append(data['labels'])

    vectorizer = CountVectorizer()
    feature_vectors = vectorizer.fit_transform(all_features).toarray()

    return feature_vectors, np.array(labels)

graph2vec_input_dir = '/content/sample_data/newgraph/dataset/graph2vec_input'  # Update this path accordingly

features_fixed, labels_fixed = preprocess_data_fixed_length(graph2vec_input_dir)
X_train_fixed, X_test_fixed, y_train_fixed, y_test_fixed = train_test_split(features_fixed, labels_fixed, test_size=0.3, random_state=42)

classes = np.unique(labels_fixed)
y_test_binarized = label_binarize(y_test_fixed, classes=classes)
y_train_binarized = label_binarize(y_train_fixed, classes=classes)

# Train classifier
classifier = OneVsRestClassifier(RandomForestClassifier(random_state=42))
classifier.fit(X_train_fixed, y_train_binarized)

# Predict probabilities
y_score = classifier.predict_proba(X_test_fixed)

fpr = dict()
tpr = dict()
roc_auc = dict()
n_classes = y_train_binarized.shape[1]
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Compute micro-average ROC curve and ROC area
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_binarized.ravel(), y_score.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])


all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

# Average it and compute AUC
mean_tpr /= n_classes
fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

# Plot all ROC curves
plt.figure()

plt.plot(fpr["micro"], tpr["micro"],
         label='Micro-average ROC curve (area = {0:0.2f})'.format(roc_auc["micro"]),
         color='deeppink', linestyle=':', linewidth=4)

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Random Forest')
plt.legend(loc="lower right")
plt.show()


Complete ROC-AUC Curve

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.feature_extraction.text import CountVectorizer

def preprocess_data_fixed_length(directory):
    all_features = []
    labels = []

    for file in os.listdir(directory):
        file_path = os.path.join(directory, file)
        with open(file_path, 'r') as json_file:
            data = json.load(json_file)
            feature_str = ' '.join([f'node{node_id}_feature{feature_value}' for node_id, feature_value in data['features'].items()])
            all_features.append(feature_str)
            labels.append(data['labels'])

    vectorizer = CountVectorizer()
    feature_vectors = vectorizer.fit_transform(all_features).toarray()

    return feature_vectors, np.array(labels)

graph2vec_input_dir = '/content/sample_data/newgraph/dataset/graph2vec_input'  # Update this path accordingly

features_fixed, labels_fixed = preprocess_data_fixed_length(graph2vec_input_dir)
X_train_fixed, X_test_fixed, y_train_fixed, y_test_fixed = train_test_split(features_fixed, labels_fixed, test_size=0.3, random_state=42)

# Binarize the labels for a multi-class problem
classes = np.unique(labels_fixed)
y_test_binarized = label_binarize(y_test_fixed, classes=classes)
y_train_binarized = label_binarize(y_train_fixed, classes=classes)

# Train classifier
classifier = OneVsRestClassifier(RandomForestClassifier(random_state=42))
classifier.fit(X_train_fixed, y_train_binarized)

# Predict probabilities
y_score = classifier.predict_proba(X_test_fixed)

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
n_classes = y_train_binarized.shape[1]
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Compute micro-average ROC curve and ROC area
fpr["micro"], tpr["micro"], _ = roc_curve(y_test_binarized.ravel(), y_score.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

# Plot all ROC curves
plt.figure(figsize=(10, 8))

# Plot ROC curve for each class
colors = ['blue', 'green', 'red', 'cyan', 'magenta', 'yellow', 'black', 'orange', 'purple', 'brown']
for i, color in zip(range(n_classes-1), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'.format(classes[i], roc_auc[i]))

# Plot micro-average ROC curve
plt.plot(fpr["micro"], tpr["micro"],
         label='Micro-average ROC curve (area = {0:0.2f})'.format(roc_auc["micro"]),
         color='deeppink', linestyle=':', linewidth=4)

# Plot the diagonal line
plt.plot([0, 1], [0, 1], 'k--', lw=2)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Each Class and Micro-average ROC Curve for Random Forest')
plt.legend(loc="lower right")
plt.savefig("roc_curve.png")
plt.show()
